In [34]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [35]:
### Extract text from pdf 

def load_pdf(data):
    loader = DirectoryLoader(
        data, 
        glob="*.pdf",
         loader_cls=PyPDFLoader
         )
    documents = loader.load()
    return documents

In [36]:
import os

print(os.getcwd())
os.chdir(r"C:\CardBot\chatBot")

extract_text = load_pdf("data")

C:\CardBot\chatBot


In [37]:
extract_text


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-19T04:25:03+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-19T04:25:03+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data\\CardBot.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='ASB Credit & Debit Cards Comparison | Data from asb.co.nz (Nov 2026)\nPage 1\n ASB Credit Cards & Visa Debit Card\n Complete Comparison Guide for Choosing the Right Card\nCompiled from official ASB website content \x7f Rates & fees current as of November 2026 \x7f Subject to change\n1. Overview\nASB offers a range of Visa credit cards designed for different needs — from the lowest interest rate option for careful\nspenders, to rewards cards for everyday purchases, and a premium Platinum card with travel insurance. The ASB Visa Debit\ncard lets you spend your own money with zero annual card fees. All 

In [38]:
len(extract_text)

3

In [39]:
from typing import List
from langchain.schema import Document

def filter_docs(docs: List[Document]) -> List[Document]:
    """Keep only each document's page content and source metadata."""
    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src},
            )
        )

    return minimal_docs

In [7]:
minimal_docs = filter_docs(extract_text)

In [40]:
## split docs into chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [41]:
text_chunks = text_split(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 146


In [42]:
text_chunks

[Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='ASB Credit & Debit Cards Comparison | Data from asb.co.nz (Nov 2026)\nPage 1'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='Page 1\n ASB Credit Cards & Visa Debit Card\n Complete Comparison Guide for Choosing the Right Card'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='Compiled from official ASB website content \x7f Rates & fees current as of November 2026 \x7f Subject to'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='2026 \x7f Subject to change'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='1. Overview'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='ASB offers a range of Visa credit cards designed for different needs — from the lowest interest'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, page_content='the lowest interest rate option for careful'),
 Document(metadata={'source': 'data\\CardBot.pdf'}, pag

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    return embeddings


    


In [ ]:
embedding = download_hugging_face_embeddings()

In [45]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [46]:
vector =embedding.embed_query("what is the meaning of life?")
vector

[-0.05871792510151863,
 0.09474470466375351,
 -0.06244673952460289,
 0.02271219529211521,
 -0.05048517510294914,
 0.0037386170588433743,
 0.09869317710399628,
 -0.015877459198236465,
 0.0985545814037323,
 0.0042405808344483376,
 0.037245746701955795,
 -0.06093670427799225,
 -0.01629030518233776,
 -0.008019961416721344,
 -0.0044191814959049225,
 -0.03710293024778366,
 -0.07821597903966904,
 -0.03404517099261284,
 -0.002461281605064869,
 -0.020916610956192017,
 0.04273826628923416,
 0.011293330229818821,
 -0.03769342601299286,
 0.05371250584721565,
 -0.09862415492534637,
 0.11139774322509766,
 -0.005809536669403315,
 0.023701820522546768,
 -0.00021257785556372255,
 0.022446271032094955,
 0.028453754261136055,
 0.053525764495134354,
 0.08441054075956345,
 -0.02425568178296089,
 0.02138282172381878,
 0.03508720546960831,
 0.09516581892967224,
 -0.03466903418302536,
 0.020812075585126877,
 0.018211020156741142,
 -0.03947438299655914,
 -0.0247601717710495,
 -0.00544181140139699,
 -0.00315222

In [47]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [48]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [49]:
from pinecone import Pinecone
pinecone_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_key)
pc


In [50]:
from pinecone import ServerlessSpec

index_name = "cardbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [51]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embedding
)

In [52]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [54]:
dswith= Document(
    page_content="This is a sample document.",
    metadata={"source": "sampleFB"}
)

In [55]:
docsearch.add_documents(documents=[dswith])

['88f718ac-2801-4c90-aea4-caa2af42a8f0']

In [56]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [57]:
retriever_docs=retriever.invoke("What is ASB Visa Rewards")
retriever_docs

[Document(id='3aa37ba5-69c7-4f26-bd8d-d071f6270b52', metadata={'source': 'data\\CardBot.pdf'}, page_content='ASB Visa Rewards — Everyday rewards with low annual fee'),
 Document(id='1f8b55bf-6310-4e03-ad44-51350e86cbc3', metadata={'source': 'data\\CardBot.pdf'}, page_content='ASB Visa Rewards — Everyday rewards with low annual fee')]

In [80]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatmodel = ChatGoogleGenerativeAI(
     model="gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    transport="rest",      # Fixes Windows gRPC hangs causing 504 errors
)
chatmodel.invoke("Hello, are you working?")

AIMessage(content="Hello! Yes, I am working and ready to help you. What's on your mind?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': []}, id='run--01a1f81a-ee9e-72bb-ba71-97021285e01d-0', usage_metadata={'input_tokens': 7, 'output_tokens': 20, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}})

In [82]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate 

In [83]:
system_prompt = (
    "You are an card assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt=ChatPromptTemplate.from_messages(
[
    ("system", system_prompt),
    ("human", "{input}")
]
)

In [84]:
question_Answer_chain = create_stuff_documents_chain(
    chatmodel,
   prompt
)
rag_chain = create_retrieval_chain(retriever,question_Answer_chain)

In [89]:
response = rag_chain.invoke({"input": "Which card is good for travel rewards?"})
print(response["answer"])

Based on the provided context, the premium Platinum card is good for travel.
